# EDA — Proyecto Integrador: Predicción de Default en Préstamos (Lending Club)

**Objetivo de esta sección:** Análisis exploratorio exhaustivo (unidimensional y bidimensional) del dataset *Lending Club Loan Data (2007–2020)*, antes de cualquier preprocesamiento o modelado.


In [ ]:
# 0. Librerías y configuración
!pip install matplotlib seaborn scipy missingno pandas numpy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import missingno as msno
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 200)

DATA_PATH = r"C:\Users\ESTUDIANTE\Downloads\accepted_2007_to_2018q4.csv\accepted_2007_to_2018q4.csv"  # <-- ajusta a tu ruta local (dataset completo, sin muestrear)
RANDOM_STATE = 42

## Carga inicial y visión general

In [ ]:
# Columnas que realmente se usan en todo el EDA
cols_necesarias = [
    "loan_status",       # para crear 'default'
    "loan_amnt", "int_rate", "annual_inc", "dti", "fico_range_high", "emp_length",
    "purpose", "home_ownership", "addr_state", "verification_status"
]

df = pd.read_csv(DATA_PATH, usecols=cols_necesarias, low_memory=False)

print(f"Dimensión del dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")
display(df.head())

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe(include=[np.number]).T

In [ ]:
df.describe(include=["object"]).T

### Creación de la variable objetivo `default`

In [ ]:
df["default"] = df["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)
df["default"].value_counts(dropna=False)

## Análisis unidimensional 

### Variables numéricas
`loan_amnt`, `int_rate`, `annual_inc`, `dti`, `fico_range_high`, `emp_length`.

In [ ]:
numeric_vars = ["loan_amnt", "int_rate", "annual_inc", "dti", "fico_range_high"]
# emp_length suele venir como texto (ej. "10+ years"); se convierte a numérico
if "emp_length" in df.columns:
    df["emp_length_num"] = (
        df["emp_length"]
        .astype(str)
        .str.extract(r"(\d+)")
        .astype(float)
    )
    numeric_vars.append("emp_length_num")

numeric_vars = [v for v in numeric_vars if v in df.columns]
numeric_vars

In [ ]:
# Estadísticos descriptivos: media, mediana, std, min, max, percentiles, IQR
desc = df[numeric_vars].describe(percentiles=[.25, .5, .75]).T
desc["IQR"] = desc["75%"] - desc["25%"]
desc

In [ ]:
# Detección de outliers por el método del IQR
def iqr_outlier_summary(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((series < lower) | (series > upper)).sum()
    pct_out = 100 * n_out / series.notna().sum()
    return pd.Series({"lower_bound": lower, "upper_bound": upper,
                       "n_outliers": n_out, "pct_outliers": pct_out})

outlier_summary = df[numeric_vars].apply(iqr_outlier_summary).T
outlier_summary

In [ ]:
# Histogramas de distribución de frecuencias
fig, axes = plt.subplots(nrows=(len(numeric_vars) + 1) // 2, ncols=2, figsize=(14, 4 * ((len(numeric_vars) + 1)//2)))
axes = axes.flatten()
for ax, col in zip(axes, numeric_vars):
    df[col].hist(bins=50, ax=ax, edgecolor="black", alpha=0.7)
    ax.set_title(f"Distribución de {col}")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots individuales (asimetría / outliers)
fig, axes = plt.subplots(nrows=(len(numeric_vars) + 1) // 2, ncols=2, figsize=(14, 4 * ((len(numeric_vars) + 1)//2)))
axes = axes.flatten()
for ax, col in zip(axes, numeric_vars):
    sns.boxplot(x=df[col], ax=ax, color="skyblue")
    ax.set_title(f"Boxplot de {col}")
plt.tight_layout()
plt.show()

In [ ]:
# Evaluación de normalidad: asimetría (skewness) y Shapiro-Wilk (muestra, dado el tamaño del dataset)
normality = pd.DataFrame(index=numeric_vars, columns=["skewness", "shapiro_stat", "shapiro_p", "sugiere_transformacion"])
for col in numeric_vars:
    s = df[col].dropna()
    skew = s.skew()
    # Shapiro-Wilk solo soporta ~5000 obs; se usa una muestra aleatoria para el test formal
    sample = s.sample(min(len(s), 5000), random_state=RANDOM_STATE)
    stat, p = stats.shapiro(sample)
    normality.loc[col] = [skew, stat, p, abs(skew) > 1]
normality

In [ ]:
# Tasa de valores faltantes por variable numérica
missing_num = df[numeric_vars].isna().mean().mul(100).sort_values(ascending=False)
missing_num.to_frame("pct_missing")

### Variables categóricas
`purpose`, `home_ownership`, `addr_state`, `verification_status`.

In [ ]:
categorical_vars = ["purpose", "home_ownership", "addr_state", "verification_status"]
categorical_vars = [v for v in categorical_vars if v in df.columns]

for col in categorical_vars:
    print(f"\n--- {col} ---")
    freq = df[col].value_counts(dropna=False)
    freq_pct = df[col].value_counts(normalize=True, dropna=False).mul(100).round(2)
    display(pd.concat([freq, freq_pct], axis=1, keys=["frecuencia_absoluta", "frecuencia_relativa_%"]))

In [ ]:
# Categorías con muy baja frecuencia (<1%) — candidatas a agruparse en "Otros"
for col in categorical_vars:
    freq_pct = df[col].value_counts(normalize=True, dropna=False).mul(100)
    low_freq = freq_pct[freq_pct < 1]
    print(f"{col}: {len(low_freq)} categorías con <1% de frecuencia -> {list(low_freq.index)}")

> **Interpretacion categorías redundantes:** Revisando las categorías de cada variable, no se identifican nombres duplicados o solapados en `purpose`, `addr_state` ni `verification_status` — cada categoría representa un concepto distinto. En `home_ownership` sí existen categorías con significado muy similar y baja frecuencia que podrían unificarse: `OTHER` y `NONE` (ambas representan situaciones de tenencia no especificada/atípica) podrían agruparse en una única categoría `OTHER`, ya que por separado tienen muy pocas observaciones y no aportan distinción real para el modelo. `ANY` también es candidata a fusionarse con `OTHER` por el mismo motivo.

In [ ]:
# Gráficos de barras por variable categórica
for col in categorical_vars:
    plt.figure(figsize=(10, 4))
    order = df[col].value_counts().index
    sns.countplot(y=df[col], order=order, color="steelblue")
    plt.title(f"Distribución de {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Tasa de valores faltantes en categóricas
missing_cat = df[categorical_vars].isna().mean().mul(100).sort_values(ascending=False)
missing_cat.to_frame("pct_missing")

### Variable objetivo (`default`)

In [ ]:
target_counts = df["default"].value_counts()
target_pct = df["default"].value_counts(normalize=True).mul(100).round(2)
display(pd.concat([target_counts, target_pct], axis=1, keys=["conteo", "porcentaje_%"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, ax=axes[0], palette=["seagreen", "indianred"])
axes[0].set_title("Distribución de la variable 'default'")
axes[0].set_xticklabels(["Fully Paid (0)", "Charged Off (1)"])

axes[1].pie(target_counts.values, labels=["Fully Paid (0)", "Charged Off (1)"],
            autopct="%1.1f%%", colors=["seagreen", "indianred"])
axes[1].set_title("Proporción de default")
plt.tight_layout()
plt.show()

> **Interpretacion:** Existe un desbalance marcado en la variable objetivo: **88.12% de los préstamos son "Fully Paid" (clase 0, n = 1,992,142)** frente a **11.88% "Charged Off" (clase 1, n = 268,559)**, una proporción cercana a 88/12, en línea con lo típico reportado para Lending Club (~80/20). Este desbalance tiene varias implicaciones directas para el modelado:
>
> - **Split de datos:** el `train_test_split` debe hacerse con `stratify=df["default"]` para preservar esta proporción 88/12 en train y test.
> - **Métricas de evaluación:** el *accuracy* es engañoso aquí (un modelo trivial que siempre prediga "Fully Paid" ya obtendría ~88% de accuracy sin aprender nada). Se deben priorizar métricas sensibles a la clase minoritaria: **F1-score, Recall/Precision de la clase 1, ROC-AUC y PR-AUC** (esta última especialmente útil dado el desbalance).
> - **Balanceo de clases:** conviene evaluar estrategias como `class_weight='balanced'` (disponible en modelos como `LogisticRegression`, `RandomForestClassifier`, etc.), sobremuestreo con **SMOTE** sobre la clase minoritaria, o submuestreo (*undersampling*) de la clase mayoritaria — aplicados siempre **después** del split y solo sobre el conjunto de entrenamiento, para evitar fuga de información hacia el test.
> - **Umbral de decisión:** también vale la pena explorar el ajuste del *threshold* de clasificación (en vez del 0.5 por defecto) según el costo de negocio de falsos negativos (prestar a alguien que hará default) vs. falsos positivos (rechazar a un buen pagador).

## Análisis bidimensional (relaciones entre variables)

### Variables numéricas vs. `default`

In [ ]:
# Boxplots comparativos por clase
fig, axes = plt.subplots(nrows=(len(numeric_vars) + 1) // 2, ncols=2, figsize=(14, 4 * ((len(numeric_vars) + 1)//2)))
axes = axes.flatten()
for ax, col in zip(axes, numeric_vars):
    sns.boxplot(x="default", y=col, data=df, ax=ax, palette=["seagreen", "indianred"])
    ax.set_title(f"{col} por clase de default")
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots (densidad por clase)
fig, axes = plt.subplots(nrows=(len(numeric_vars) + 1) // 2, ncols=2, figsize=(14, 4 * ((len(numeric_vars) + 1)//2)))
axes = axes.flatten()
for ax, col in zip(axes, numeric_vars):
    sns.violinplot(x="default", y=col, data=df, ax=ax, palette=["seagreen", "indianred"])
    ax.set_title(f"Densidad de {col} por clase")
plt.tight_layout()
plt.show()

In [ ]:
# Diferencia de medias por clase + prueba estadística (t-test o Mann-Whitney) + correlación punto-biserial
results = []
for col in numeric_vars:
    g0 = df.loc[df["default"] == 0, col].dropna()
    g1 = df.loc[df["default"] == 1, col].dropna()

    mean_diff = g1.mean() - g0.mean()

    # Normalidad aproximada (sobre muestra) para decidir el test
    _, p_norm = stats.shapiro(g0.sample(min(len(g0), 5000), random_state=RANDOM_STATE))
    if p_norm > 0.05:
        stat, p_val = stats.ttest_ind(g0, g1, equal_var=False)
        test_used = "t-test (Welch)"
    else:
        stat, p_val = stats.mannwhitneyu(g0, g1, alternative="two-sided")
        test_used = "Mann-Whitney U"

    # Correlación punto-biserial
    valid = df[[col, "default"]].dropna()
    r_pb, p_pb = stats.pointbiserialr(valid["default"], valid[col])

    results.append({
        "variable": col, "media_clase0": g0.mean(), "media_clase1": g1.mean(),
        "diferencia_medias": mean_diff, "test": test_used, "estadistico": stat,
        "p_value": p_val, "significativo_0.05": p_val < 0.05,
        "corr_punto_biserial": r_pb, "p_corr": p_pb
    })

bivar_numeric = pd.DataFrame(results).set_index("variable")
bivar_numeric

### Variables categóricas vs. `default`

In [ ]:
# Tablas de contingencia (tasa de default por categoría) + chi-cuadrado
chi2_results = []
for col in categorical_vars:
    ct = pd.crosstab(df[col], df["default"])
    ct_pct = pd.crosstab(df[col], df["default"], normalize="index").mul(100).round(2)
    print(f"\n--- Tasa de default por categoría de '{col}' ---")
    display(ct_pct)

    chi2, p, dof, _ = stats.chi2_contingency(ct)
    chi2_results.append({"variable": col, "chi2": chi2, "dof": dof, "p_value": p,
                          "significativo_0.05": p < 0.05})

chi2_summary = pd.DataFrame(chi2_results).set_index("variable")
chi2_summary

In [ ]:
# Barras apiladas/agrupadas: frecuencia de default por categoría
for col in categorical_vars:
    ct_pct = pd.crosstab(df[col], df["default"], normalize="index").mul(100)
    ct_pct.plot(kind="bar", stacked=True, figsize=(10, 4), color=["seagreen", "indianred"])
    plt.title(f"Tasa de default (%) por categoría de {col}")
    plt.ylabel("% de préstamos")
    plt.legend(["Fully Paid", "Charged Off"])
    plt.tight_layout()
    plt.show()

> **Interpretacion:** Las cuatro variables categóricas resultaron **estadísticamente significativas** en la prueba chi-cuadrado (p < 0.05 en todos los casos, con p-valores prácticamente 0): `purpose` (χ² = 5,506.64, gl = 13), `home_ownership` (χ² = 6,040.70, gl = 5), `addr_state` (χ² = 3,226.07, gl = 50) y `verification_status` (χ² = 20,391.76, gl = 2), esta última con el estadístico más alto, lo que sugiere una asociación particularmente fuerte con `default`.
>
> En cuanto a las categorías con tasas de default más altas:
>
> - **`purpose`:** `small_business` presenta la tasa más alta con diferencia (**18.55%**), seguida de `renewable_energy` (15.29%) y `moving` (14.37%). En el otro extremo, `car` (8.92%) y `credit_card` (9.67%) son las más seguras.
> - **`home_ownership`:** `OTHER` (14.84%) y `RENT` (13.86%) muestran las tasas más altas, mientras que `ANY` (5.62%) y `MORTGAGE` (10.30%) las más bajas — quienes tienen hipoteca o vivienda propia tienden a pagar mejor que quienes arriendan.
> - **`verification_status`:** contraintuitivamente, `Verified` tiene la tasa de default **más alta** (15.85%), por encima de `Source Verified` (12.33%) y `Not Verified` (7.99%). Esto podría deberse a que Lending Club verifica con mayor frecuencia a los solicitantes que ya presentan un perfil de mayor riesgo.
> - **`addr_state`:** hay dispersión notable entre estados, desde `ME` (5.65%) e `IA` (7.14%) en el extremo bajo, hasta `AL` (14.39%) y `AR` (14.17%) en el extremo alto — una diferencia de ~9 puntos porcentuales entre el estado más seguro y el más riesgoso.

### Multicolinealidad entre variables independientes

In [ ]:
# Matriz de correlación de Pearson entre numéricas
corr_matrix = df[numeric_vars].corr(method="pearson")

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Matriz de correlación (Pearson) — variables numéricas")
plt.tight_layout()
plt.show()

# Pares con correlación alta (|r| > 0.7)
high_corr = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .rename("correlacion")
    .reset_index()
)
high_corr = high_corr[high_corr["correlacion"].abs() > 0.7]
high_corr

In [ ]:
# V de Cramer entre variables categóricas (asociación)
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    return np.sqrt((chi2 / n) / (min(r - 1, k - 1)))

cramers_matrix = pd.DataFrame(index=categorical_vars, columns=categorical_vars, dtype=float)
for c1 in categorical_vars:
    for c2 in categorical_vars:
        cramers_matrix.loc[c1, c2] = cramers_v(df[c1].astype(str), df[c2].astype(str))

plt.figure(figsize=(6, 5))
sns.heatmap(cramers_matrix.astype(float), annot=True, fmt=".2f", cmap="Purples", vmin=0, vmax=1)
plt.title("Asociación entre categóricas (V de Cramer)")
plt.tight_layout()
plt.show()

> **Interpretacion redundancias:** No se detectaron pares de variables con correlación fuerte (|r| > 0.7); la tabla `high_corr` resultó vacía, por lo que no hay evidencia de multicolinealidad severa entre las variables numéricas seleccionadas. La correlación más notable de la matriz es entre `fico_range_high` e `int_rate` (**r = -0.42**), una relación negativa moderada consistente con la intuición de negocio: a mayor score crediticio, menor tasa de interés asignada por Lending Club. Aunque no es lo suficientemente alta como para justificar excluir una de las dos variables por redundancia, sí conviene tenerla presente en modelos lineales (ej. regresión logística), donde ambas podrían competir por explicar la misma señal de riesgo. El resto de correlaciones son débiles (todas con |r| < 0.20): `annual_inc` con `loan_amnt` (r = 0.20) es la segunda más alta, seguida de `dti` con `int_rate` (r = 0.12). En conjunto, esto sugiere que las variables numéricas aportan información relativamente independiente entre sí, sin necesidad de eliminar ninguna por colinealidad.

### Visualizaciones avanzadas

In [ ]:
# Pairplot — ÚNICA excepción autorizada al uso de muestra (solo para visualización)
sample_df = df.sample(n=min(5000, len(df)), random_state=RANDOM_STATE)
sns.pairplot(sample_df, vars=numeric_vars[:5], hue="default",
             palette={0: "seagreen", 1: "indianred"}, diag_kind="kde", plot_kws={"alpha": 0.4, "s": 15})
plt.suptitle("Pairplot (muestra de 5,000 obs.) coloreado por default", y=1.02)
plt.show()

In [ ]:
# Gráficos de densidad (KDE) comparativos por clase — variables más relevantes
top_vars = bivar_numeric["corr_punto_biserial"].abs().sort_values(ascending=False).head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for ax, col in zip(axes, top_vars):
    sns.kdeplot(data=df, x=col, hue="default", fill=True, common_norm=False,
                palette={0: "seagreen", 1: "indianred"}, ax=ax)
    ax.set_title(f"Densidad de {col} por clase")
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de dispersión (scatter matrix) limitada a variables clave
from pandas.plotting import scatter_matrix
scatter_matrix(sample_df[numeric_vars[:5]], figsize=(12, 12), diagonal="kde", alpha=0.4)
plt.suptitle("Scatter matrix (muestra de 5,000 obs.)", y=1.02)
plt.show()

## Valores faltantes

In [ ]:
missing_full = df.isna().sum().to_frame("n_missing")
missing_full["pct_missing"] = (missing_full["n_missing"] / len(df) * 100).round(2)
missing_full.sort_values("pct_missing", ascending=False)

In [ ]:
# Mapa de patrones de nulos (missingno)
cols_with_missing = missing_full[missing_full["pct_missing"] > 0].index.tolist()
if cols_with_missing:
    msno.matrix(df[cols_with_missing].sample(min(len(df), 20000), random_state=RANDOM_STATE))
    plt.title("Patrón de valores faltantes (muestra de 20,000 filas para visualización)")
    plt.show()

    msno.heatmap(df[cols_with_missing])
    plt.title("Correlación entre patrones de nulos")
    plt.show()

In [ ]:
# ¿Relación entre missingness y la variable objetivo? (aprox. MCAR vs no-MCAR)
mcar_check = []
for col in cols_with_missing:
    is_missing = df[col].isna().astype(int)
    ct = pd.crosstab(is_missing, df["default"])
    if ct.shape[0] == 2:
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        mcar_check.append({"variable": col, "chi2": chi2, "p_value": p,
                            "asociado_con_default": p < 0.05})

pd.DataFrame(mcar_check).set_index("variable") if mcar_check else "Sin columnas con missing values a evaluar." 

In [ ]:
# Plan de tratamiento propuesto por variable (a completar según resultados anteriores)
plan_tratamiento = missing_full.copy()
def sugerir_tratamiento(pct):
    if pct == 0:
        return "sin acción"
    elif pct > 70:
        return "eliminar columna"
    elif pct > 30:
        return "evaluar: imputación avanzada o creación de categoría 'Missing'"
    else:
        return "imputar (mediana/moda) o categoría 'Missing'"

plan_tratamiento["tratamiento_sugerido"] = plan_tratamiento["pct_missing"].apply(sugerir_tratamiento)
plan_tratamiento.sort_values("pct_missing", ascending=False)

## Resumen ejecutivo del EDA

| Aspecto | Hallazgo |
|---|---|
| **Calidad de los datos** | El dataset está mayormente limpio: la mayoría de columnas numéricas y categóricas clave tienen missing casi nulo (0.00%–0.08%), salvo `emp_length`/`emp_length_num` con **6.50%** de faltantes. No se encontraron celdas con errores de tokenización ni tipos inconsistentes tras la carga con `usecols`. |
| **Variables más prometedoras para predecir `default`** | `int_rate` es la variable numérica más asociada al target (correlación punto-biserial **r = 0.199**, la más alta de todas), seguida de `fico_range_high` (**r = -0.119**, relación inversa esperada: a mayor score, menor riesgo). Entre las categóricas, `verification_status` mostró la asociación más fuerte en la prueba chi-cuadrado (**χ² = 20,391.76**), seguida de `home_ownership` y `purpose` — todas con p < 0.05. |
| **Problemas detectados** | **Desbalance de clases:** 88.12% Fully Paid vs. 11.88% Charged Off (~88/12). · **Outliers severos en:** `annual_inc` (4.87% de outliers por IQR), `fico_range_high` (3.31%), `int_rate` (1.82%) y `loan_amnt` (1.56%). · **Alta correlación entre:** ninguna supera |r| > 0.7; la más notable es `fico_range_high` con `int_rate` (r = -0.42, moderada, sin justificar exclusión). · **Missing values sistemáticos en:** `emp_length` (6.50%) y `dti` (0.08%), ambos **asociados estadísticamente con `default`** (chi-cuadrado p < 0.05 en el check MCAR), por lo que no son missing completamente al azar (MAR/MNAR) y su tratamiento debe hacerse con cuidado. |
| **Decisiones preliminares de preprocesamiento** | **Variables a mantener:** las 6 numéricas (`loan_amnt`, `int_rate`, `annual_inc`, `dti`, `fico_range_high`, `emp_length_num`) y las 4 categóricas (`purpose`, `home_ownership`, `addr_state`, `verification_status`), ya que ninguna resultó redundante por colinealidad y todas mostraron asociación significativa con el target. · **Transformaciones necesarias:** `annual_inc` (asimetría = 493.9) y `dti` (asimetría = 29.2) presentan asimetría extrema y no-normalidad (Shapiro p ≈ 0), por lo que se recomienda transformación logarítmica o `RobustScaler`; `fico_range_high` también se beneficiaría de una transformación más leve (asimetría = 1.19). · **Tratamiento de nulos:** imputar `emp_length`/`emp_length_num` y `dti` con mediana/moda o, dado que su ausencia está asociada al target, considerar crear una categoría/indicador binario `is_missing` en vez de imputación simple; el resto de columnas (missing < 0.01%) no requiere acción, basta con eliminar esas pocas filas. · **¿Se requiere balanceo de clases?:** Sí — usar `stratify` en el split, métricas ponderadas (F1, ROC-AUC/PR-AUC) y evaluar `class_weight='balanced'` o SMOTE sobre el set de entrenamiento. |